# Beginner's Guide to Exploratory Data Analysis (EDA) in Google Colab
**Dataset:** `Sample_Superstore.csv`

Welcome to **Exploratory Data Analysis (EDA)**! Think of EDA like being a detective arriving at a new scene: before jumping to conclusions or building complex models, we examine the clues, check for missing pieces, measure what is typical, and draw visual maps to uncover hidden stories.

### How to Use This Notebook in Google Colab
1. **Upload the CSV:** Click the **Folder icon** on the left sidebar of Google Colab, then click the **Upload** icon (page with an up arrow) and select `Sample_Superstore.csv`.
2. **Run Cells Step-by-Step:** Click on a code cell and press **`Shift + Enter`** (or click the Play button on the left of the cell) to run it and move to the next one.

### Our 6-Step EDA Roadmap
1. **Setup & Data Loading:** Importing Python's data toolkit and loading our table.
2. **First Look & Data Profiling:** Checking rows, columns, data types, missing values, and duplicates.
3. **Data Cleaning & Type Formatting:** Fixing date formats and creating helpful new features.
4. **Univariate Analysis:** Examining one column at a time (Mean vs. Median, spread, and outliers).
5. **Bivariate & Multivariate Analysis:** Comparing two or more columns to find relationships and business drivers.
6. **Key Takeaways & Next Steps:** Summarizing what the data tells us.

---
## Step 1: Setup & Loading the Dataset

Python uses pre-built toolkits called **libraries** so we don't have to write everything from scratch:
* **`pandas` (`pd`):** The ultimate spreadsheet tool for Python. It loads data into a table called a **DataFrame (`df`)**.
* **`numpy` (`np`):** A math engine for working with numbers and arrays.
* **`matplotlib.pyplot` (`plt`) & `seaborn` (`sns`):** Our charting libraries for creating visual graphs.

> **Beginner Tip:** Notice `encoding='latin1'` inside `pd.read_csv()` below. Some CSV files exported from Excel contain special characters (like accented letters in customer names) that require `'latin1'` encoding instead of the default `'utf-8'` so Python doesn't throw a `UnicodeDecodeError`.

In [ ]:
# 1. Import our essential data analysis and visualization libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set a clean, modern visual theme for all our charts
sns.set_theme(style="whitegrid", palette="muted")

# 2. Load the Sample_Superstore.csv file into a Pandas DataFrame
# Make sure you have uploaded 'Sample_Superstore.csv' to your Colab session files!
df = pd.read_csv('Sample_Superstore.csv', encoding='latin1')

# Confirm the data loaded properly
print("Success! Dataset loaded into a DataFrame called 'df'.")

---
## Step 2: First Look & Data Profiling (Structure & Quality)

Before doing any math, we need to inspect the structure and health of our dataset. This process is called **Data Profiling**:
* **`df.shape`**: Tells us `(rows, columns)` — how big our table is.
* **`df.head()`**: Shows the first 5 rows so we can see actual examples of the data.
* **`df.info()`**: Lists every column name, how many non-empty values it has, and its **data type** (`int64` = whole number, `float64` = decimal number, `object` = text/string).
* **`df.isnull().sum()` & `df.duplicated().sum()`**: Checks for missing blanks and exact duplicate rows.

In [ ]:
# Check the dimensions of the dataset (Rows, Columns)
rows, cols = df.shape
print(f"Dataset Dimensions: {rows:,} rows and {cols} columns\n")

# Display the first 5 rows to see what the raw data looks like
print("--- First 5 Rows ---")
display(df.head())

In [ ]:
# Inspect column names, non-null counts, and data types
print("--- Column Info & Data Types ---")
df.info()

# Check for missing values in each column
print("\n--- Missing Values Per Column ---")
print(df.isnull().sum())

# Check for completely identical duplicate rows
duplicate_count = df.duplicated().sum()
print(f"\nTotal Duplicate Rows: {duplicate_count}")

---
## Step 3: Data Cleaning & Feature Engineering

Look closely at the `df.info()` output above:
1. **`Order Date` and `Ship Date`** are currently stored as `object` (plain text), not true calendar dates! Converting them with `pd.to_datetime()` lets Python sort chronologically, extract years/months, and calculate shipping times.
2. **`Postal Code`** is stored as an integer (`int64`), even though zip codes are geographic labels rather than numbers we would ever add or average.
3. **Feature Engineering:** We can create brand-new columns from existing ones to unlock deeper insights—such as **`Shipping_Days`** (how long delivery took) and **`Profit_Margin`** (profit divided by sales).

In [ ]:
# 1. Convert text date columns into true Pandas datetime objects
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

# 2. Convert Postal Code to text (string) so it isn't treated as a math metric
df['Postal Code'] = df['Postal Code'].astype(str)

# 3. Feature Engineering: Create helpful new analytical columns
# Calculate how many days elapsed between ordering and shipping
df['Shipping_Days'] = (df['Ship Date'] - df['Order Date']).dt.days

# Extract the Order Year and Order Month for time-trend analysis
df['Order_Year'] = df['Order Date'].dt.year
df['Order_Month'] = df['Order Date'].dt.to_period('M').astype(str)

# Calculate Profit Margin (%) for each line item
df['Profit_Margin'] = (df['Profit'] / df['Sales']) * 100

print("Data types updated and new features created!")
display(df[['Order Date', 'Ship Date', 'Shipping_Days', 'Sales', 'Profit', 'Profit_Margin']].head())

---
## Step 4: Univariate Analysis (Examining One Variable at a Time)

"Univariate" simply means looking at **one column in isolation**. We examine two types of variables differently:

### 4A. Numerical Variables (Central Tendency, Spread & Outliers)
* **Mean (Average):** Sensitive to extreme values (outliers).
* **Median (50% mark):** The exact middle value when sorted—resistant to outliers.
* **Standard Deviation (`std`):** Measures how spread out the numbers are from the mean.
* **IQR & Box Plots:** A Box Plot shows the middle 50% of data inside a box (`25%` to `75%`) and flags extreme outliers as individual dots beyond the whiskers (`1.5 * IQR`).

In [ ]:
# Generate summary statistics for our key numerical columns
numeric_cols = ['Sales', 'Quantity', 'Discount', 'Profit', 'Shipping_Days', 'Profit_Margin']
print("--- Summary Statistics (Numerical Columns) ---")
# .T transposes (flips) the table so columns are easier to read as rows
display(df[numeric_cols].describe().T.round(2))

In [ ]:
# Visualize the distribution and outliers of 'Sales' and 'Profit'
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# 1. Histogram of Sales (zoomed in under $1,000 so the main pattern is visible)
sns.histplot(df[df['Sales'] < 1000]['Sales'], bins=40, kde=True, ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Distribution of Sales (Orders Under $1,000)')
axes[0, 0].set_xlabel('Sales ($)')

# 2. Box Plot of Sales (shows extreme high-value outliers)
sns.boxplot(x=df['Sales'], ax=axes[0, 1], color='steelblue')
axes[0, 1].set_title('Box Plot of All Sales (Highlighting Outliers)')
axes[0, 1].set_xlabel('Sales ($)')

# 3. Histogram of Profit (zoomed between -$200 and +$200)
sns.histplot(df[(df['Profit'] > -200) & (df['Profit'] < 200)]['Profit'], bins=40, kde=True, ax=axes[1, 0], color='seagreen')
axes[1, 0].axvline(0, color='red', linestyle='--', label='Break-Even ($0)')
axes[1, 0].set_title('Distribution of Profit (-$200 to +$200)')
axes[1, 0].set_xlabel('Profit ($)')
axes[1, 0].legend()

# 4. Box Plot of Profit (shows both massive losses and massive wins)
sns.boxplot(x=df['Profit'], ax=axes[1, 1], color='seagreen')
axes[1, 1].set_title('Box Plot of All Profit (Losses & Gains)')
axes[1, 1].set_xlabel('Profit ($)')

plt.tight_layout()
plt.show()

### 4B. Categorical Variables (Frequency Counts)
For text categories like `Segment`, `Region`, `Category`, and `Ship Mode`, we can't calculate an average. Instead, we count how often each category appears using **Bar Charts** (`sns.countplot`).

In [ ]:
# Plot order counts across four key categorical dimensions
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

categorical_cols = ['Category', 'Segment', 'Region', 'Ship Mode']
titles = ['Orders by Product Category', 'Orders by Customer Segment', 'Orders by Region', 'Orders by Shipping Mode']

# Loop through each category and draw a bar chart in its grid position
for ax, col, title in zip(axes.flatten(), categorical_cols, titles):
    order = df[col].value_counts().index  # Sort bars from highest to lowest
    sns.countplot(data=df, x=col, order=order, ax=ax, hue=col, palette='viridis', legend=False)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Number of Line Items')

    # Add exact count labels on top of each bar
    for container in ax.containers:
        ax.bar_label(container, padding=3)

plt.tight_layout()
plt.show()

---
## Step 5: Bivariate & Multivariate Analysis (Relationships & Business Drivers)

Now comes the heart of data analytics: comparing **two or more variables together** to answer real business questions!
1. **Categorical vs. Numerical (`groupby`):** Which product categories and sub-categories generate the most Sales vs. Profit? Are any losing money?
2. **Numerical vs. Numerical (Correlation & Scatter Plots):** How does offering a `Discount` affect `Profit`? (Remember: *Correlation measures linear strength from -1 to +1, and correlation does NOT automatically prove causation!*)
3. **Time Series Trend:** How have monthly sales evolved over time?

In [ ]:
# Group data by Category to compare Total Sales, Total Profit, and Average Discount
category_summary = df.groupby('Category')[['Sales', 'Profit', 'Quantity']].sum().reset_index()
category_summary['Overall_Margin_%'] = (category_summary['Profit'] / category_summary['Sales']) * 100

print("--- Performance by Product Category ---")
display(category_summary.round(2))

# Visualize Total Sales vs. Total Profit across all 17 Sub-Categories
subcat_summary = df.groupby('Sub-Category')[['Sales', 'Profit']].sum().sort_values(by='Profit', ascending=False).reset_index()

plt.figure(figsize=(12, 6))
# Color bars green if profitable, red if losing money overall
colors = ['seagreen' if p >= 0 else 'crimson' for p in subcat_summary['Profit']]
sns.barplot(data=subcat_summary, x='Profit', y='Sub-Category', palette=colors, hue='Sub-Category', legend=False)
plt.axvline(0, color='black', linewidth=1)
plt.title('Total Profit by Sub-Category (Green = Profitable, Red = Net Loss)', fontsize=13, fontweight='bold')
plt.xlabel('Total Profit ($)')
plt.ylabel('Sub-Category')
plt.tight_layout()
plt.show()

In [ ]:
# 1. Correlation Heatmap across numerical columns
plt.figure(figsize=(8, 5))
corr_matrix = df[['Sales', 'Quantity', 'Discount', 'Profit', 'Shipping_Days']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Numerical Variables', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# 2. Scatter Plot: How does Discount level impact Profit across Categories?
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Discount', y='Profit', hue='Category', alpha=0.6)
plt.axhline(0, color='red', linestyle='--', linewidth=1.2, label='Break-Even ($0)')
plt.title('Impact of Discount Rate on Profit by Category', fontsize=13, fontweight='bold')
plt.xlabel('Discount Rate (0.20 = 20% Off)')
plt.ylabel('Profit ($)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Group by Order_Month to track monthly Sales and Profit over time
monthly_trends = df.groupby('Order_Month')[['Sales', 'Profit']].sum().reset_index()

plt.figure(figsize=(14, 5))
plt.plot(monthly_trends['Order_Month'], monthly_trends['Sales'], marker='o', label='Monthly Sales ($)', color='steelblue')
plt.plot(monthly_trends['Order_Month'], monthly_trends['Profit'], marker='s', label='Monthly Profit ($)', color='seagreen')

# Show every 3rd month label on the X-axis so text doesn't overlap
plt.xticks(ticks=range(0, len(monthly_trends), 3), labels=monthly_trends['Order_Month'][::3], rotation=45)
plt.title('Monthly Sales and Profit Trend (2014–2017)', fontsize=13, fontweight='bold')
plt.xlabel('Order Month')
plt.ylabel('USD ($)')
plt.legend()
plt.tight_layout()
plt.show()

---
## Step 6: Key EDA Takeaways & Beginner Practice Challenge

### What Our EDA Uncovered in `Sample_Superstore.csv`
1. **Clean Structure:** The dataset has **9,994 rows and 21 columns** with **zero missing values** and **zero duplicate rows**, though `Order Date` and `Ship Date` required conversion from text to datetime format.
2. **Right-Skewed Distributions:** Most individual order lines are small (median sale is **~$54.49**, median profit is **~$8.67**), while a small number of high-ticket corporate/technology orders pull the average (`mean`) much higher (**~$229.86**).
3. **Sub-Category Profit Leaks:** While **Copiers, Phones, and Accessories** drive strong profits, **Tables, Bookcases, and Supplies** generate a net financial loss overall.
4. **Discount Sensitivity:** The correlation heatmap and scatter plot show a negative relationship (`r = -0.22`) between `Discount` and `Profit`—heavy discounts (above 20–30%) frequently push orders into negative profit territory.
5. **Seasonal Spikes:** Monthly sales show recurring spikes every **November and December** (Q4 holiday/end-of-year purchasing).

---

### Your Turn: Beginner Practice Challenge
Try writing your own code in the empty cell below to answer these two business questions:
1. **Region Comparison:** Which `Region` (West, East, Central, South) has the highest total `Profit`, and what is the average `Discount` given in each region? *(Hint: Use `df.groupby('Region')[['Sales', 'Profit', 'Discount']].mean()` or `.sum()`)*
2. **Segment Visualization:** Create a bar chart comparing total `Sales` across the three `Segment` groups (`Consumer`, `Corporate`, `Home Office`).

In [ ]:
# Practice Exercise: Write your own Pandas code here!
# Example starter:
# region_analysis = df.groupby('Region')[['Sales', 'Profit', 'Discount']].agg({'Sales': 'sum', 'Profit': 'sum', 'Discount': 'mean'})
# display(region_analysis)
